# Bulk-loading parcel polygons into SQL Server (chunk 9)

The same pipeline as `Parcels_import.ipynb`, applied to rows `200000:225000` of
the parcel dataset. Kept separate because each chunk was run and verified
independently during the import.

Credentials are read from environment variables — see `.env.example`.


In [ ]:
# import the needed libraries
import pandas as pd
import geopandas as gpd
import pyodbc
from shapely.geometry import Polygon, MultiPolygon
from shapely.geometry import mapping
import json
from geoalchemy2 import Geometry, WKTElement
import sqlalchemy as sal
from tqdm import tqdm
# from multiprocessing import Pool, cpu_count

In [ ]:
# reading the table that contains the attributes of the parcels, it is has been simplified in ArcGIS Pro Pro before exported in a csv file
table = pd.read_csv('data_table1.csv')
table.shape[0]

In [ ]:
# reading the first chunk that exported from the source dataset as a GeoDataFrame
# it is exported from ArcGIS Pro with removing the Z dimention from the geometry
#p1 = gpd.read_file('parcels.gdb', driver="OpenFileGDB", layer='chunk_1_100000_v2')
# p2 = gpd.read_file('parcels.gdb', driver="OpenFileGDB", layer='chunk_2_100000_v2')
# p3 = gpd.read_file('parcels.gdb', driver="OpenFileGDB", layer='chunk_3_50000')
# p4 = gpd.read_file('parcels.gdb', driver="OpenFileGDB", layer='chunk_4_50000')
# p5 = gpd.read_file('parcels.gdb', driver="OpenFileGDB", layer='chunk_5_100000')
# p6 = gpd.read_file('parcels.gdb', driver="OpenFileGDB", layer='chunk_6_100000')
# p8 = gpd.read_file('parcels.gdb', driver="OpenFileGDB", layer='chunk_7_200000')[0:100000]
#p8 = gpd.read_file('parcels.gdb', driver="OpenFileGDB", layer='chunk_7_200000')[100000:]
# p9 = gpd.read_file('parcels.gdb', driver="OpenFileGDB", layer='chunk_9_100000')
#p10 = gpd.read_file('parcels.gdb', driver="OpenFileGDB", layer='chunk_10_100000')
# p11 = gpd.read_file('parcels.gdb', driver="OpenFileGDB", layer='chunk_11_250000')
p12 = gpd.read_file('parcels.gdb', driver="OpenFileGDB", layer='chunk_12_246739')
p12.shape[0]

In [ ]:
# create a new column for SourceObjectID and fill it with sequenced numbers
#p1['SourceObjectID'] = range(100001, len(p1)+1)
# p2['SourceObjectID'] = range(100001, 100001 + 100000)
# p3['SourceObjectID'] = range(200001, 200001 + 50000)
# p4['SourceObjectID'] = range(250001, 250001 + 50000)
# p5['SourceObjectID'] = range(300001, 300001 + 100000)
#p6['SourceObjectID'] = range(400001, 400001 + 100000)
#p7['SourceObjectID'] = range(500001, 500001 + 100000)
# p8['SourceObjectID'] = range(600001, 600001 + 100000)
#p9['SourceObjectID'] = range(700001, 700001 + 100000)
#p10['SourceObjectID'] = range(800001, 800001 + 100000)
#p11['SourceObjectID'] = range(900001, 900001 + 250000)
p12['SourceObjectID'] = range(1150001, 1150001 + 246739)

In [ ]:
# define the bad/empty geometries
empty_geometry_rows = p12[p12['geometry'].is_empty]

In [ ]:
# print the count of the empty geometries
empty_geometry_rows.shape[0]

In [ ]:
# drop the empty geometries from the current chunk/GeoDataFrame
p12 = p12.drop(empty_geometry_rows.index)
p12.shape[0]

In [ ]:
# join the csv table with the current parcels chunk/GeoDataFrame to get all of the needed columns
merged = pd.merge(p12, table, how='left', on=['CleanParcelID', 'FullLegalDescription', 'FullOwnerName', 'FullOwnerAddress', 'FullPhysicalAddress'])

In [ ]:
merged.shape[0]

In [ ]:
merged.columns

In [ ]:
merged.tail(2)

In [ ]:
# drop some columns that are not needed to be loaded to the database
merged = merged.drop(columns=['CleanParcelID','OID_','Shape_Length','Shape_Area'])

# rename the remain columns to match the same names of the table in the database
merged.rename(columns={'ID':'SourceGISID', 'ParcelID':'SourceFullParcelID', 'DistrictCode':'DistrictID',
                      'TaxDistrict':'TaxMapID', 'Suffix':'SuffixID', 'FullOwnerName':'SourceFullOwnerName',
                      'FullOwnerAddress':'SourceFullOwnerAddress', 'FullPhysicalAddress':'SourceFullPhysicalAddress'}, inplace=True)

In [ ]:
# assign values of SourceFullParcelID column to ParcelID column
merged['ParcelID'] = merged['SourceFullParcelID']
merged.rename(columns={'Label':'SourceLabelName'}, inplace=True)

In [ ]:
# change the crs of the GeoDataFrame to EPSG:3857
merged = merged.to_crs('EPSG:3857')

In [ ]:
# create a new empty string column [GeoJSON] to carry all of the values of all of the columns in a GeoJSON format later
merged['GeoJSON'] = ''

In [ ]:
# assign the name of the source shapefile/dataset to the column DataSource to know the source of the data
merged['DataSource'] = 'WV_Parcels_All_Counties'

In [ ]:
# create a 3 new string empty columns, these columns will be filled on the databse using get_date() SQL function
merged[['SourceDateTime','SysStartTime','SysEndTime']] = ''

In [ ]:
# clean the NaN values by replacing them to 0
merged[['SourceGISID','TaxMapID','DistrictID']] = merged[['SourceGISID','TaxMapID','DistrictID']].fillna(0).astype(int)

In [ ]:
# replace the 0 values with '' after converting the 3 columns into string
merged[['SourceGISID','TaxMapID','DistrictID']] = merged[['SourceGISID','TaxMapID','DistrictID']].astype(str)
merged[['SourceGISID','TaxMapID','DistrictID']] = merged[['SourceGISID','TaxMapID','DistrictID']].replace('0','')

In [ ]:
# design a new columns order to make the GeoDataFrame matches the order of the table on the database
order = ['SourceObjectID','SourceGISID','SourceFullParcelID','CountyID','DistrictID','TaxMapID','ParcelID','SuffixID','FullLegalDescription',
         'SourceFullOwnerName','SourceFullOwnerAddress','SourceFullPhysicalAddress','SourceLabelName','GeoJSON','DataSource','SourceDateTime',
        'SysStartTime','SysEndTime','geometry']

# convert these columns into string to make them match the data types of the table on the database
merged[['SourceObjectID','SourceGISID','DistrictID','TaxMapID']] = merged[['SourceObjectID','SourceGISID','DistrictID','TaxMapID']].astype(str)

In [ ]:
# assign the designed order to the GeoDataFrame
merged = merged[order]

In [ ]:
# clean the data by replace nan values with ''
merged = merged.replace('nan', '')
merged = merged.fillna('')
# merged.head()

In [ ]:
for index, row in merged.iterrows():
    properties = row.drop('geometry').to_dict()  # Exclude the geometry column
    geometry = mapping(row['geometry'])  # Convert MultiPolygon to GeoJSON-like dictionary
    feature = {"type": "Feature","properties": properties,"geometry": geometry}
    # Update the 'geojson' column with the GeoJSON string
    merged.at[index, 'GeoJSON'] = json.dumps(feature)

In [ ]:
# create a new string geometry column and fill it with the values of the actual column of the GeoDataFrame
merged['geom'] = ''
for idx, row in merged.iterrows():
    merged.at[idx, 'geom'] = str(row['geometry'])

In [ ]:
# checking for the large values of GeoJSON column
merged['length'] = merged['GeoJSON'].str.len()
top_10 = merged.nlargest(10, 'length')
# top_10

In [ ]:
large_geojson = merged[merged['length'] >= 200000]
# large_geojson

In [ ]:
# big_geoms = gpd.read_file('bad value.geojson')
# big_geoms = big_geoms.append(large_geojson)
# big_geoms.to_file('bad value.geojson')
large_geojson.shape[0]

In [ ]:
# ids = ['862319','883900']
# large_geojson = merged[merged['SourceObjectID'].isin(ids)]
# large_geojson
merged.drop(large_geojson.index, inplace=True)
merged.shape[0]

In [ ]:
merged = merged.drop(columns='length')
merged.columns

In [ ]:
# Database connection details are read from environment variables so that no
# credentials are stored in this notebook. See .env.example for the full list.
import os

server = os.environ['DB_SERVER']
database = os.environ['DB_NAME']
username = os.environ['DB_USER']
password = os.environ['DB_PASSWORD']


In [ ]:
# start building the connection to access the database and load the data to the dedicated table
cnxn = pyodbc.connect('DRIVER={SQL Server};SERVER='+server+';DATABASE='+database+';UID='+username+';PWD='+ password)
cursor = cnxn.cursor()

In [ ]:
# create a new GeoDataFrame to slice it and load data in chunks
gdf = merged.drop(columns='geometry')[200000:225000]
# r = ['590027','590028','590029']
# gdf = merged[merged['SourceObjectID'].isin(r)]
# اخر رقم في السطر اللي فوق مش بيتحسب ... خليك فاكر
gdf.columns

In [ ]:
# assign the order of the columns to the new GeoDataFrame
gdf_order = ['SourceObjectID', 'SourceGISID', 'SourceFullParcelID', 'CountyID',
       'DistrictID', 'TaxMapID', 'ParcelID', 'SuffixID',
       'FullLegalDescription', 'SourceFullOwnerName', 'SourceFullOwnerAddress',
       'SourceFullPhysicalAddress', 'SourceLabelName', 'GeoJSON', 'geom', 'DataSource',
       'SourceDateTime', 'SysStartTime', 'SysEndTime']

gdf = gdf[gdf_order]
gdf.columns

In [ ]:
gdf.shape[0]

In [ ]:
# the SQL query that will load the data to the dedicated table on the database
sql_query = "INSERT INTO dbo.WVTaxParcels_Import (SourceObjectID,SourceGISID,SourceFullParcelID,CountyID,DistrictID,TaxMapID,ParcelID,SuffixID,FullLegalDescription,SourceFullOwnerName,SourceFullOwnerAddress,SourceFullPhysicalAddress,SourceLabelName,GeoJSON,Geom,DataSource,SourceDateTime,SysStartTime,SysEndTime) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,geometry::STGeomFromText(?, 3857),?,?,?,?)"

In [ ]:
# define the chunk size as 1000 rows
data = [tuple(row) for row in gdf.itertuples(index=False)]
chunk_size = 1000
total_rows = len(data)

In [ ]:
# iterate on the chunks and execute the SQL query to the chunks
with tqdm(total=total_rows, desc="Inserting rows") as pbar:
    for chunk in range(0, total_rows, chunk_size): 
        chunk_data = data[chunk:chunk+chunk_size]
        cursor.executemany(sql_query, chunk_data)
        pbar.update(len(chunk_data))

cnxn.commit()

cursor.close()